# 01 — Data Discovery
WiDS Datathon 2025 ADHD dataset — real data inventory.
Target: `ADHD_Outcome`. `Sex_F` is excluded from all modeling per project scope.

In [1]:

import os, time
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
PROJECT_ROOT = '/home/claude/adhd_project'
os.chdir(PROJECT_ROOT)
BASE = 'data/raw/widsdatathon2025'
print('Base exists:', os.path.exists(BASE))


Base exists: True


## Load raw files (TRAIN_NEW — most recent, 36P Pearson connectome)

In [2]:

t0 = time.time()
train_cat = pd.read_excel(f'{BASE}/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx')
train_quan = pd.read_excel(f'{BASE}/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx')
train_sol = pd.read_excel(f'{BASE}/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx')
train_fcm = pd.read_csv(f'{BASE}/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv')
test_cat = pd.read_excel(f'{BASE}/TEST/TEST_CATEGORICAL.xlsx')
test_quan = pd.read_excel(f'{BASE}/TEST/TEST_QUANTITATIVE_METADATA.xlsx')
test_fcm = pd.read_csv(f'{BASE}/TEST/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv')
print(f'Loaded in {time.time()-t0:.1f}s')


Loaded in 9.9s


In [3]:

inventory = []
def describe(name, df):
    inventory.append({
        'file': name,
        'rows': df.shape[0],
        'cols': df.shape[1],
        'memory_MB': round(df.memory_usage(deep=True).sum()/1e6, 2),
    })

describe('TRAIN_CATEGORICAL_METADATA_new.xlsx', train_cat)
describe('TRAIN_QUANTITATIVE_METADATA_new.xlsx', train_quan)
describe('TRAINING_SOLUTIONS.xlsx', train_sol)
describe('TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv', train_fcm)
describe('TEST_CATEGORICAL.xlsx', test_cat)
describe('TEST_QUANTITATIVE_METADATA.xlsx', test_quan)
describe('TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv', test_fcm)

inv_df = pd.DataFrame(inventory)
inv_df


,file,rows,cols,memory_MB
0,TRAIN_CATEGORICAL_METADATA_new.xlsx,1213,10,0.16
1,TRAIN_QUANTITATIVE_METADATA_new.xlsx,1213,19,0.25
2,TRAINING_SOLUTIONS.xlsx,1213,3,0.09
3,TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_P...,1213,19901,193.18
4,TEST_CATEGORICAL.xlsx,304,10,0.04
5,TEST_QUANTITATIVE_METADATA.xlsx,304,19,0.06
6,TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv,304,19901,48.42


## Subject ID alignment across files

In [4]:

ids_cat = set(train_cat['participant_id'])
ids_quan = set(train_quan['participant_id'])
ids_sol = set(train_sol['participant_id'])
ids_fcm = set(train_fcm['participant_id'])

print('cat == sol:', ids_cat == ids_sol)
print('quan == sol:', ids_quan == ids_sol)
print('fcm == sol:', ids_fcm == ids_sol)
print('N train subjects (solutions):', len(ids_sol))
print('N duplicate subject ids in sol:', train_sol['participant_id'].duplicated().sum())
print('N duplicate rows in fcm:', train_fcm.duplicated().sum())


cat == sol: True
quan == sol: True
fcm == sol: True
N train subjects (solutions): 1213
N duplicate subject ids in sol: 0


N duplicate rows in fcm: 0


## ADHD target distribution

In [5]:

adhd_counts = train_sol['ADHD_Outcome'].value_counts().sort_index()
adhd_pct = train_sol['ADHD_Outcome'].value_counts(normalize=True).sort_index() * 100
print('ADHD_Outcome counts:')
print(adhd_counts)
print()
print('ADHD_Outcome percentages:')
print(adhd_pct.round(2))
print()
print('Class imbalance ratio (majority:minority) = %.2f : 1' % (adhd_counts.max()/adhd_counts.min()))


ADHD_Outcome counts:
ADHD_Outcome
0    382
1    831
Name: count, dtype: int64

ADHD_Outcome percentages:
ADHD_Outcome
0    31.49
1    68.51
Name: proportion, dtype: float64

Class imbalance ratio (majority:minority) = 2.18 : 1


## Feature groups: connectome vs metadata

In [6]:

connectome_cols = [c for c in train_fcm.columns if c != 'participant_id']
quan_cols = [c for c in train_quan.columns if c != 'participant_id']
cat_cols = [c for c in train_cat.columns if c != 'participant_id']

print('Connectome (functional connectivity edges):', len(connectome_cols))
print('Quantitative metadata columns:', len(quan_cols), quan_cols)
print('Categorical metadata columns:', len(cat_cols), cat_cols)


Connectome (functional connectivity edges): 19900
Quantitative metadata columns: 18 ['EHQ_EHQ_Total', 'ColorVision_CV_Score', 'APQ_P_APQ_P_CP', 'APQ_P_APQ_P_ID', 'APQ_P_APQ_P_INV', 'APQ_P_APQ_P_OPD', 'APQ_P_APQ_P_PM', 'APQ_P_APQ_P_PP', 'SDQ_SDQ_Conduct_Problems', 'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Emotional_Problems', 'SDQ_SDQ_Externalizing', 'SDQ_SDQ_Generating_Impact', 'SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Internalizing', 'SDQ_SDQ_Peer_Problems', 'SDQ_SDQ_Prosocial', 'MRI_Track_Age_at_Scan']
Categorical metadata columns: 9 ['Basic_Demos_Enroll_Year', 'Basic_Demos_Study_Site', 'PreInt_Demos_Fam_Child_Ethnicity', 'PreInt_Demos_Fam_Child_Race', 'MRI_Track_Scan_Location', 'Barratt_Barratt_P1_Edu', 'Barratt_Barratt_P1_Occ', 'Barratt_Barratt_P2_Edu', 'Barratt_Barratt_P2_Occ']


## Data quality checks

In [7]:

quality = {}
quality['fcm_missing_any'] = bool(train_fcm[connectome_cols].isnull().values.any())
quality['fcm_inf_any'] = bool(np.isinf(train_fcm[connectome_cols].values).any())
quality['fcm_constant_cols'] = int((train_fcm[connectome_cols].nunique() <= 1).sum())
quality['quan_missing_total'] = int(train_quan[quan_cols].isnull().sum().sum())
quality['cat_missing_total'] = int(train_cat[cat_cols].isnull().sum().sum())
quality['fcm_duplicate_rows'] = int(train_fcm.duplicated().sum())
quality['fcm_duplicate_subjects'] = int(train_fcm['participant_id'].duplicated().sum())

for k, v in quality.items():
    print(f'{k}: {v}')

print()
print('Missing values per quantitative column:')
print(train_quan[quan_cols].isnull().sum()[train_quan[quan_cols].isnull().sum() > 0])
print()
print('Missing values per categorical column:')
print(train_cat[cat_cols].isnull().sum()[train_cat[cat_cols].isnull().sum() > 0])


fcm_missing_any: False
fcm_inf_any: False
fcm_constant_cols: 0
quan_missing_total: 549
cat_missing_total: 566
fcm_duplicate_rows: 0
fcm_duplicate_subjects: 0

Missing values per quantitative column:
EHQ_EHQ_Total                  13
ColorVision_CV_Score           23
APQ_P_APQ_P_CP                 12
APQ_P_APQ_P_ID                 12
APQ_P_APQ_P_INV                12
APQ_P_APQ_P_OPD                12
APQ_P_APQ_P_PM                 12
APQ_P_APQ_P_PP                 12
SDQ_SDQ_Conduct_Problems        9
SDQ_SDQ_Difficulties_Total      9
SDQ_SDQ_Emotional_Problems      9
SDQ_SDQ_Externalizing           9
SDQ_SDQ_Generating_Impact       9
SDQ_SDQ_Hyperactivity           9
SDQ_SDQ_Internalizing           9
SDQ_SDQ_Peer_Problems           9
SDQ_SDQ_Prosocial               9
MRI_Track_Age_at_Scan         360
dtype: int64

Missing values per categorical column:
PreInt_Demos_Fam_Child_Ethnicity     43
PreInt_Demos_Fam_Child_Race          54
MRI_Track_Scan_Location               3
Barratt_Barratt_

## Connectome structure (correlation edges)

In [8]:

# 19900 = C(200,2) -> 200 ROIs (Regions of Interest), i.e. upper triangle of a 200x200 correlation matrix
import math
n_edges = len(connectome_cols)
n_rois = (1 + math.isqrt(1 + 8*n_edges)) // 2
print(f'N edges: {n_edges}')
print(f'Implied N ROIs (from n(n-1)/2 = {n_edges}): {n_rois}')
print(f'Check: {n_rois}*({n_rois}-1)/2 = {n_rois*(n_rois-1)//2}')

sample_vals = train_fcm[connectome_cols].values.astype('float32')
print()
print('Edge value range: [%.4f, %.4f]' % (sample_vals.min(), sample_vals.max()))
print('Edge value mean: %.4f, std: %.4f' % (sample_vals.mean(), sample_vals.std()))
print('Consistent with Pearson correlations (bounded [-1,1]):', bool(sample_vals.min() >= -1.0001 and sample_vals.max() <= 1.0001))

edge_var = train_fcm[connectome_cols].var()
print()
print('Edge variance distribution:')
print(edge_var.describe())


N edges: 19900
Implied N ROIs (from n(n-1)/2 = 19900): 200
Check: 200*(200-1)/2 = 19900



Edge value range: [-0.8754, 0.9590]
Edge value mean: 0.0227, std: 0.2513
Consistent with Pearson correlations (bounded [-1,1]): True



Edge variance distribution:
count    19900.000000
mean         0.035975
std          0.008079
min          0.006368
25%          0.030193
50%          0.034409
75%          0.040416
max          0.123547
dtype: float64


## Save dataset inventory

In [9]:

os.makedirs('reports', exist_ok=True)
inv_df.to_csv('reports/dataset_inventory.csv', index=False)

summary = pd.DataFrame([{
    'n_train_subjects': len(ids_sol),
    'n_test_subjects_unlabeled': test_fcm.shape[0],
    'n_connectome_edges': len(connectome_cols),
    'implied_n_rois': n_rois,
    'n_quantitative_features': len(quan_cols),
    'n_categorical_features': len(cat_cols),
    'adhd_positive': int(adhd_counts[1]),
    'adhd_negative': int(adhd_counts[0]),
    'adhd_positive_pct': round(float(adhd_pct[1]), 2),
    'class_imbalance_ratio': round(float(adhd_counts.max()/adhd_counts.min()), 2),
    'fcm_missing_values': quality['fcm_missing_any'],
    'fcm_constant_columns': quality['fcm_constant_cols'],
    'duplicate_subjects': quality['fcm_duplicate_subjects'],
}])
summary.to_csv('reports/dataset_discovery_summary.csv', index=False)
summary.T


,0
n_train_subjects,1213
n_test_subjects_unlabeled,304
n_connectome_edges,19900
implied_n_rois,200
n_quantitative_features,18
n_categorical_features,9
adhd_positive,831
adhd_negative,382
adhd_positive_pct,68.51
class_imbalance_ratio,2.18
